# Synthetic NIAH Counting v5

One shared v2-style GPT-2 LM with learned absolute positional embeddings and a mixed thinking/non-thinking toggle.

## Colab Setup

In [ ]:
import importlib
import os
import pathlib
import subprocess
import sys

REPO_URL = "https://github.com/Twist-Shan/Synthetic_CoT_NiaH_Count.git"
REPO_DIR = pathlib.Path("/content/Synthetic_CoT_NiaH_Count")
INSTALL_FULL_REQUIREMENTS = False  # Usually keep False on Colab; full pins can break the runtime ABI.
INSTALL_MINIMAL_DEPS = False       # Set True only if the import check below fails.
INSTALL_EDITABLE_PACKAGE = False   # Not needed when the notebook cwd is the repo root.
REPAIR_NUMPY_ABI = True        # Repair pandas/numpy/scipy binary mismatch, then restart once.

if "google.colab" in sys.modules and not REPO_DIR.exists():
    subprocess.check_call(["git", "clone", REPO_URL, str(REPO_DIR)])
if "google.colab" in sys.modules and REPO_DIR.exists():
    os.chdir(REPO_DIR)

print("cwd =", pathlib.Path.cwd())


def patch_v5_eager_attention():
    path = pathlib.Path("synthetic_niah_v5/model.py")
    if not path.exists():
        print("v5 eager-attention patch skipped; file not found:", path)
        return
    text = path.read_text(encoding="utf-8")
    if 'cfg.setdefault("attn_implementation", "eager")' in text:
        print("v5 eager-attention patch already present.")
        return
    old = '    pad_token_id = int(cfg.pop("pad_token_id", eos_token_id))\n    return GPT2Config(\n'
    new = '    pad_token_id = int(cfg.pop("pad_token_id", eos_token_id))\n    cfg.setdefault("attn_implementation", "eager")\n    return GPT2Config(\n'
    if old not in text:
        print("v5 eager-attention patch not applied; source pattern did not match.")
        return
    path.write_text(text.replace(old, new), encoding="utf-8")
    print("Patched synthetic_niah_v5/model.py to use eager attention for output_attentions.")

patch_v5_eager_attention()



def ensure_science_stack_abi():
    """Colab can break pandas/scipy if numpy is upgraded/downgraded mid-session."""
    try:
        import numpy as np
        import pandas as pd
        import scipy
        import seaborn as sns
        print(f"numpy={np.__version__} pandas={pd.__version__} scipy={scipy.__version__} seaborn={sns.__version__}")
        return
    except (ImportError, ValueError, AttributeError) as exc:
        message = str(exc)
        abi_markers = ["numpy.dtype size changed", "_ARRAY_API", "compiled using NumPy 1.x"]
        if not any(marker in message for marker in abi_markers):
            raise
        print("Detected numpy/pandas/scipy ABI mismatch:", message)
        if not REPAIR_NUMPY_ABI:
            raise RuntimeError(
                "Set REPAIR_NUMPY_ABI = True in this setup cell, rerun it, then restart the kernel."
            ) from exc
        subprocess.run(
            [
                sys.executable,
                "-m",
                "pip",
                "install",
                "-q",
                "--force-reinstall",
                "numpy<2",
                "pandas",
                "scipy",
                "scikit-learn",
                "matplotlib",
                "seaborn",
            ],
            check=True,
        )
        raise RuntimeError(
            "Repaired numpy/pandas/scipy ABI packages. Restart the Colab/VSCode kernel, then rerun this setup cell."
        ) from exc

ensure_science_stack_abi()

if INSTALL_FULL_REQUIREMENTS:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
elif INSTALL_MINIMAL_DEPS:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.41", "tqdm", "pandas", "matplotlib", "seaborn", "scikit-learn",
    ])
if INSTALL_EDITABLE_PACKAGE:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"])

for module_name in ["torch", "transformers", "sklearn", "matplotlib"]:
    importlib.import_module(module_name)
print("Dependency import check passed.")


## Runtime Settings

In [3]:
PRESET = 'debug'  # 'debug' for artifact smoke test, 'main' for full run
STAGE = 'all'
DEVICE = 'cuda'  # change to 'cpu' if needed
OUT_ROOT = 'outputs/v5'
RUN_NAME = ''
ABLATE_NO_CONFLICT_MASK = False
TRACE_INDICES = False

args = [sys.executable, '-m', 'synthetic_niah_v5.run_v5', '--preset', PRESET, '--stage', STAGE, '--device', DEVICE, '--out-root', OUT_ROOT]
if RUN_NAME:
    args += ['--run-name', RUN_NAME]
if ABLATE_NO_CONFLICT_MASK:
    args += ['--ablate-no-conflict-mask']
if TRACE_INDICES:
    args += ['--trace-indices']
print(' '.join(args))

/usr/bin/python3 -m synthetic_niah_v5.run_v5 --preset debug --stage all --device cuda --out-root outputs/v5


## Run Pipeline

In [ ]:
import subprocess

print(' '.join(args), flush=True)
proc = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
captured = []
for line in proc.stdout:
    print(line, end='')
    captured.append(line.rstrip())
returncode = proc.wait()
if returncode:
    print('---- Last 120 log lines ----')
    print('\n'.join(captured[-120:]))
    raise subprocess.CalledProcessError(returncode, args)

RUN_DIR = pathlib.Path(OUT_ROOT) / RUN_NAME if RUN_NAME else pathlib.Path(OUT_ROOT)
RUN_DIR


## Key Tables

In [ ]:
import pandas as pd

train_log = pd.read_csv(RUN_DIR / 'tables' / 'train_log.csv')
eval_by_step = pd.read_csv(RUN_DIR / 'tables' / 'eval_by_step.csv')
ambiguous = pd.read_csv(RUN_DIR / 'tables' / 'ambiguous_prefix.csv')
display(train_log.tail())
display(eval_by_step.tail(12))
display(ambiguous.tail())

: 

## Key Figures

In [ ]:
from IPython.display import Image, display

for name in [
    'train_loss_by_step_and_mode.png',
    'final_accuracy_by_step_mode.png',
    'final_accuracy_by_count_mode.png',
    'trace_metrics_by_count.png',
    'ambiguous_prefix_probs_by_step.png',
]:
    path = RUN_DIR / 'figures' / name
    if path.exists():
        display(Image(filename=str(path)))

: 

## Save to Google Drive

In [ ]:
DRIVE_SAVE_COMPLETED = False
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = pathlib.Path('/content/drive/MyDrive/synthetic_niah_v5')
    drive_dir.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(['bash', '-lc', f'cp -r {RUN_DIR} {drive_dir}/'])
    DRIVE_SAVE_COMPLETED = True
    print('Saved to', drive_dir)
else:
    print('Not in Colab; skipping Google Drive save.')

: 

## Auto-disconnect Colab Runtime

This cell runs immediately after the Google Drive save cell. It only disconnects when a Drive save was confirmed; local VSCode/Jupyter runs are not force-closed by default.

In [ ]:
AUTO_DISCONNECT_AFTER_DRIVE_SAVE = True
FORCE_LOCAL_KERNEL_SHUTDOWN = False

if AUTO_DISCONNECT_AFTER_DRIVE_SAVE and globals().get("DRIVE_SAVE_COMPLETED", False):
    import time

    print("Google Drive save completed. Flushing Drive and disconnecting Colab runtime in 3 seconds...")
    time.sleep(3)
    try:
        from google.colab import drive, runtime

        try:
            drive.flush_and_unmount()
            print("Google Drive flushed and unmounted.")
        except Exception as e:
            print(f"Drive flush/unmount skipped or failed: {e}")
        runtime.unassign()
    except Exception as e:
        print(f"Colab runtime disconnect unavailable or failed: {e}")
        if FORCE_LOCAL_KERNEL_SHUTDOWN:
            import IPython

            IPython.Application.instance().kernel.do_shutdown(restart=False)
        else:
            print("Not forcing local kernel shutdown.")
else:
    print("Auto-disconnect skipped: no confirmed Google Drive save, or AUTO_DISCONNECT_AFTER_DRIVE_SAVE is False.")

: 

## Optional GitHub Save

In [ ]:
COMMIT_RESULTS = False
if COMMIT_RESULTS:
    subprocess.check_call(['git', 'status', '--short'])
    subprocess.check_call(['git', 'add', str(RUN_DIR)])
    subprocess.check_call(['git', 'commit', '-m', f'Add synthetic NIAH v5 {PRESET} results'])
    subprocess.check_call(['git', 'push'])

: 